# Lab 3a — THE LINE
**هندسة تطبيقات النماذج اللغوية الكبيرة**  
**إعداد: أ. لطيفة سعد**

## ⓪ PROJECT SETTINGS — إعدادات مشروعك

In [1]:
ASSISTANT_NAME = "THE LINE"

SERVICES = [
    "energy_efficiency",
    "water_management",
    "sustainable_materials",
    "smart_infrastructure",
    "other"
]

CITIES = [
    "Tabuk",
    "NEOM",
    "Sharma",
    "Duba",
    "unknown"
]

print("Assistant:", ASSISTANT_NAME)
print("Services :", SERVICES)
print("Cities   :", CITIES)

Assistant: THE LINE
Services : ['energy_efficiency', 'water_management', 'sustainable_materials', 'smart_infrastructure', 'other']
Cities   : ['Tabuk', 'NEOM', 'Sharma', 'Duba', 'unknown']


## ① ServiceTicket — شكل التذكرة

In [2]:
from pydantic import BaseModel, field_validator, ValidationError

class ServiceTicket(BaseModel):
    full_name: str
    service_type: str
    city: str
    urgent: bool
    employee_id: str | None = None

    @field_validator("service_type")
    @classmethod
    def check_service(cls, value):
        if value not in SERVICES:
            raise ValueError("service_type is not supported")
        return value

    @field_validator("city")
    @classmethod
    def check_city(cls, value):
        if value not in CITIES:
            raise ValueError("city is not supported")
        return value

    @field_validator("employee_id")
    @classmethod
    def check_employee_id(cls, value):
        if value is None:
            return value
        if len(value) != 8 or not value.startswith("ENG") or not value[3:].isdigit():
            raise ValueError("employee_id must start with ENG followed by 5 digits")
        return value

## ② VALIDATION TEST — تجربة التحقق

In [3]:
ticket = ServiceTicket(
    full_name="Awadh",
    service_type="energy_efficiency",
    city="Tabuk",
    urgent=True,
    employee_id="ENG10234"
)

print(ticket)

full_name='Awadh' service_type='energy_efficiency' city='Tabuk' urgent=True employee_id='ENG10234'


In [4]:
try:
    ticket = ServiceTicket(
        full_name="Awadh",
        service_type="energy_efficiency",
        city="London",
        urgent=True,
        employee_id="12345"
    )
    print(ticket)
except ValidationError as e:
    for error in e.errors():
        print(error["loc"][0], "→", error["msg"])

city → Value error, city is not supported
employee_id → Value error, employee_id must start with ENG followed by 5 digits


## ③ extract_ticket — الاستخراج والإصلاح

In [5]:
MESSAGES = [
    "أنا عواض من تبوك، أحتاج مراجعة استهلاك الطاقة في مبنى مستدام",
    "أنا فهد، أحتاج مراجعة استخدام المياه المعالجة في مشروع داخل نيوم",
    "أنا سعد، أحتاج استشارة عن مواد مستدامة لموقع في لندن"
]

DEMO_ANSWERS = {
    MESSAGES[0]: [("Awadh", "energy_efficiency", "Tabuk")],
    MESSAGES[1]: [("Fahad", "water_management", "Neom"),
                  ("Fahad", "water_management", "NEOM")],
    MESSAGES[2]: [("Saad", "sustainable_materials", "London")]
}

def demo_model(message, errors=None):
    answers = DEMO_ANSWERS[message]
    if errors is None:
        name, service, city = answers[0]
    else:
        name, service, city = answers[-1]
    return {
        "full_name": name,
        "service_type": service,
        "city": city,
        "urgent": False
    }

print(demo_model(MESSAGES[0]))

{'full_name': 'Awadh', 'service_type': 'energy_efficiency', 'city': 'Tabuk', 'urgent': False}


In [6]:
def extract_ticket(message):
    errors = None
    for attempt in [1, 2]:
        raw = demo_model(message, errors)
        try:
            ticket = ServiceTicket(**raw)
            if attempt == 1:
                return ticket, "first_try"
            return ticket, "after_repair"
        except ValidationError as e:
            errors = e.errors()
            print("Attempt", attempt, "failed →", errors[0]["msg"])
    return None, "escalated"

In [7]:
ticket, status = extract_ticket(MESSAGES[1])

print(status)
print(ticket)

Attempt 1 failed → Value error, city is not supported
after_repair
full_name='Fahad' service_type='water_management' city='NEOM' urgent=False employee_id=None


## ④ TEST AND PASS RATE — التجربة على الرسائل ونسبة النجاح

In [8]:
for message in MESSAGES:
    ticket, status = extract_ticket(message)
    print(status, "|", message)

first_try | أنا عواض من تبوك، أحتاج مراجعة استهلاك الطاقة في مبنى مستدام
Attempt 1 failed → Value error, city is not supported
after_repair | أنا فهد، أحتاج مراجعة استخدام المياه المعالجة في مشروع داخل نيوم
Attempt 1 failed → Value error, city is not supported
Attempt 2 failed → Value error, city is not supported
escalated | أنا سعد، أحتاج استشارة عن مواد مستدامة لموقع في لندن
